# 🇰🇬 Обучение своей модели AkylAi

Этот ноутбук обучает **твою собственную ИИ-модель** на данных проекта (кыргызский язык, УК КР, ПДД, Конституция, культура, туризм, факты о Кыргызстане). Базовая модель — открытая **Qwen2.5-3B**.

## Как запускать (важно!)
1. Вверху: **Среда выполнения → Сменить среду выполнения → выбери `T4 GPU` → Сохранить**.
2. Нажимай ▶ на каждом блоке **по порядку, сверху вниз**. Жди зелёную галочку ✅, потом жми следующий.
3. Всё бесплатно. Обучение занимает ~30–60 минут.

❗ Если увидишь **красную ошибку** — скопируй её текст и пришли Claude, он подскажет что делать. Не закрывай вкладку во время обучения.

## Блок 1. Установка инструментов ⚙️
Скачивает библиотеки для обучения. Займёт ~3–5 минут. Пока идёт — просто жди.

In [ ]:
%%capture
# Устанавливаем unsloth — она сама подтянет совместимые версии trl/transformers
!pip install unsloth

## Блок 2. Проверка видеокарты (GPU) 🖥️
Убедимся, что включён бесплатный GPU. Если увидишь **Tesla T4** — всё отлично.

Если напишет «GPU не найден» — вверху: **Среда выполнения → Сменить среду выполнения → T4 GPU**, и запусти этот блок снова.

In [ ]:
import torch
if torch.cuda.is_available():
    print('✅ GPU включён:', torch.cuda.get_device_name(0))
else:
    print('❌ GPU не найден! Включи: Среда выполнения → Сменить среду выполнения → T4 GPU, потом запусти этот блок снова.')

## Блок 3. Загрузка твоего датасета с GitHub 📥
Скачивает данные проекта и собирает их в один файл. В конце покажет, **сколько примеров** получилось (сейчас около 850).

In [ ]:
!rm -rf Cloud-Code
!git clone --depth 1 https://github.com/5005os/Cloud-Code.git
!python Cloud-Code/training/prepare_dataset.py
print('---')
!wc -l Cloud-Code/training/akylai_dataset.jsonl

## Блок 4. Загрузка базовой модели (Qwen2.5-3B) 🧠
Скачивает открытую модель, на основе которой обучаем свою. Займёт несколько минут (модель ~2 ГБ).

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print('✅ Модель загружена и готова к обучению.')

## Блок 5. Подготовка данных 📝
Превращает вопросы-ответы в формат, понятный модели. В конце покажет один пример — это нормально.

In [ ]:
from datasets import load_dataset

SYSTEM = "Ты — AkylAi, кыргызский ИИ-ассистент. Отвечай точно про Кыргызстан (КР): кыргызский язык, законы КР, ПДД, Конституцию, историю, культуру и туризм. Никогда не отвечай про законы других стран."

dataset = load_dataset("json", data_files="Cloud-Code/training/akylai_dataset.jsonl", split="train")

def format_chat(ex):
    user = ex["instruction"]
    if ex.get("input"):
        user += "\n" + ex["input"]
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": user},
        {"role": "assistant", "content": ex["output"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

dataset = dataset.map(format_chat)
print('Примеров для обучения:', len(dataset))
print('--- пример ---')
print(dataset[0]["text"][:500])

## Блок 6. Обучение модели 🔥 (самый долгий, ~30–50 минут)
Запусти и **жди**. Будут бежать строчки с числом `loss` — это нормально, чем меньше число, тем лучше учится модель. Не закрывай вкладку.

In [ ]:
from trl import SFTTrainer, SFTConfig
import torch

# Новый способ (для свежих версий trl): токенизатор передаётся как processing_class,
# а настройки данных — внутри SFTConfig. Так больше нет ошибки с 'tokenizer'.
trainer = SFTTrainer(
    model = model,
    train_dataset = dataset,
    processing_class = tokenizer,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)
trainer.train()
print('🎉 Обучение завершено!')

## Блок 7. Проверка — поговори со своей моделью 💬
Модель ответит на тестовые вопросы. Можешь поменять текст в кавычках и спросить своё.

In [ ]:
FastLanguageModel.for_inference(model)

def ask(question):
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": question},
    ]
    inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
    out = model.generate(input_ids=inputs, max_new_tokens=256, temperature=0.7)
    print('❓', question)
    print('🤖', tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))
    print('---')

ask("Как будет 'спасибо' на кыргызском?")
ask("Что предусматривает статья 122 УК КР?")
ask("Что посмотреть туристу в Кыргызстане?")
ask("Расскажи про эпос Манас.")

## Блок 8. Сохранение модели 💾
Сохраняет твою модель: сначала LoRA-адаптер, потом GGUF-файл (для запуска в Ollama на своём компьютере). Если GGUF не соберётся — не страшно, сохраним в другом формате.

In [ ]:
# LoRA-адаптер (маленький файл со «знаниями»)
model.save_pretrained("akylai_lora")
tokenizer.save_pretrained("akylai_lora")
print('✅ LoRA сохранён в папку akylai_lora')

# GGUF — чтобы запускать в Ollama / llama.cpp на своём компьютере
try:
    model.save_pretrained_gguf("akylai_model", tokenizer, quantization_method="q4_k_m")
    print('✅ GGUF готов: папка akylai_model (файл .gguf)')
except Exception as e:
    print('⚠️ GGUF не собрался:', e)
    print('Ничего страшного — сохраняю модель в обычном формате (16bit).')
    model.save_pretrained_merged("akylai_model", tokenizer, save_method="merged_16bit")
    print('✅ Модель сохранена в папку akylai_model (формат 16bit)')

## Блок 9. Скачать модель на компьютер ⬇️
Упакует модель в один архив `akylai_model.zip` и начнёт скачивание в браузер (файл появится в «Загрузках»). Это и есть **твоя модель AkylAi**.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("akylai_model", "zip", "akylai_model")
print('✅ Архив готов, начинаю скачивание...')
files.download("akylai_model.zip")

## 🎉 Готово! У тебя своя модель AkylAi

Что дальше:
- **Запустить на своём компьютере** через [Ollama](https://ollama.com): распакуй архив, рядом с файлом `.gguf` создай файл `Modelfile` с одной строкой `FROM ./akylai_model.gguf`, затем в терминале:
  ```
  ollama create akylai -f Modelfile
  ollama run akylai
  ```
- **Сделать модель умнее** — пополняй папку `dataset/` в проекте новыми данными и запусти этот ноутбук снова.

Сделано для Кыргызстана 🇰🇬